<a href="https://colab.research.google.com/github/zy538324/BlueTeam/blob/main/LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

matt8638_cybsec_training_path = kagglehub.dataset_download('matt8638/cybsec-training')

print('Data source import complete.')


In [ ]:
# Install all necessary tokenization libraries and upgrade transformers first
!pip install --upgrade transformers -qq
!pip install tokenizers -qq
!pip install tiktoken -qq
!pip install sentencepiece -qq

print("Required libraries installed/upgraded. Please restart the runtime now (Runtime > Restart runtime) and then run all cells.")

In [ ]:
import numpy as np
import pandas as pd
import os

# Using the path provided by kagglehub
dataset_path = matt8638_cybsec_training_path

print(f"Listing files in: {dataset_path}")
for dirname, _, filenames in os.walk(dataset_path):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
import json
import os
import logging
from typing import Dict, List, Optional, Tuple

import torch
from datasets import Dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import LabelEncoder
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments

# 1. Setup Standard Logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

def extract_pentest_phase(text: str) -> str:
    """Extract the pentest phase based on keywords."""
    phases = {
        "reconnaissance": ["reconnaissance", "information gathering", "footprinting"],
        "scanning": ["scanning", "enumeration", "vulnerability assessment"],
        "gaining_access": ["exploitation", "gaining access", "privilege escalation"],
        "maintaining_access": ["persistence", "maintaining access", "backdoor"],
        "covering_tracks": ["covering tracks", "log cleaning", "evidence removal"],
    }

    text = text.lower()
    for phase, keywords in phases.items():
        if any(keyword in text for keyword in keywords):
            return phase
    return "other"

def normalize_text_value(value: Optional[str]) -> str:
    if value is None:
        return ""
    return str(value).strip()

def append_record(texts: List[str], labels: List[str], qa_pairs: List[Dict[str, str]], text: str, answer: str = "") -> None:
    clean_text = normalize_text_value(text)
    clean_answer = normalize_text_value(answer)
    if not clean_text:
        return

    texts.append(clean_text)
    labels.append(extract_pentest_phase(clean_text))
    qa_pairs.append({
        "question": clean_text,
        "answer": clean_answer or "Can you share more detail about the target system and your specific security objective?",
    })

def load_and_preprocess_data() -> Tuple[List[str], List[str], List[Dict[str, str]]]:
    """Load and preprocess data dynamically from the downloaded dataset path."""
    texts = []
    labels = []
    qa_pairs = []

    # Use the variable from the kagglehub download
    json_files = []
    target_dir = matt8638_cybsec_training_path

    for dirname, _, filenames in os.walk(target_dir):
        for filename in filenames:
            if filename.endswith('.json'):
                json_files.append(os.path.join(dirname, filename))

    if not json_files:
        raise FileNotFoundError(f"No JSON files were found in '{target_dir}'.")

    for json_path in json_files:
        logger.info(f"Loading data from {json_path}")
        with open(json_path, "r", encoding="utf-8") as f:
            data_json = json.load(f)
            if isinstance(data_json, list):
                for entry in data_json:
                    if isinstance(entry, dict):
                        text_candidate = entry.get("description", "") or entry.get("question", "") or entry.get("content", "")
                        answer_candidate = entry.get("response", "") or entry.get("answer", "") or entry.get("remediation", "")
                        append_record(texts, labels, qa_pairs, text_candidate, answer_candidate)

    if not texts:
        raise ValueError("No training texts were loaded. Please check the structure of your JSON files.")

    logger.info(f"Successfully loaded {len(texts)} records.")
    return texts, labels, qa_pairs

# ... (rest of the helper functions remain the same) ...

def get_all_phases(texts: List[str]) -> List[str]:
    phases = set()
    for text in texts:
        phases.add(extract_pentest_phase(text))
    return sorted(list(phases))

def prepare_dataset(texts: List[str], labels: List[str], label_encoder: LabelEncoder, tokenizer: AutoTokenizer) -> Dataset:
    encoded_labels = label_encoder.transform(labels)
    penetration_encodings = tokenizer(texts, truncation=True, padding=True, max_length=512)
    return Dataset.from_dict({
        "input_ids": penetration_encodings["input_ids"],
        "attention_mask": penetration_encodings["attention_mask"],
        "labels": torch.tensor(encoded_labels)
    })

def train_model(model: AutoModelForSequenceClassification, dataset: Dataset, output_dir: str):
    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=3,
        per_device_train_batch_size=8, # Increased batch size
        fp16=True, # Enabled mixed precision training
        warmup_steps=100,
        weight_decay=0.01,
        logging_dir="/content/logs",
        report_to="none",
        save_total_limit=4
    )
    trainer = Trainer(model=model, args=training_args, train_dataset=dataset)
    trainer.train()

def main():
    logger.info("Starting data processing...")
    texts, labels, qa_pairs = load_and_preprocess_data()
    unique_phases = get_all_phases(texts)

    label_encoder = LabelEncoder()
    label_encoder.fit(unique_phases)

    tokenizer = AutoTokenizer.from_pretrained("roberta-large")
    model = AutoModelForSequenceClassification.from_pretrained("roberta-large", num_labels=len(unique_phases))

    dataset = prepare_dataset(texts, labels, label_encoder, tokenizer)
    output_dir = "/content/model_output"
    train_model(model, dataset, output_dir)

    # Explicitly save the trained model and tokenizer after training
    logger.info(f"Saving trained model and tokenizer to {output_dir}")
    model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    logger.info("Model and tokenizer saved successfully.")

# To run the process, call main()
main()

In [ ]:
!pip install sentencepiece -qq

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from sklearn.preprocessing import LabelEncoder

# Ensure all necessary tokenization libraries and transformers are up-to-date
!pip install --upgrade transformers -qq
!pip install tokenizers -qq
!pip install tiktoken -qq
!pip install sentencepiece -qq

# Load the trained model and tokenizer
loaded_tokenizer = AutoTokenizer.from_pretrained("/content/model_output")
loaded_model = AutoModelForSequenceClassification.from_pretrained("/content/model_output")

# Get the label encoder from the training phase
# Note: In a real deployment, you'd save/load the label_encoder too.
# For this demonstration, we'll recreate it based on the unique_phases detected during training.

# This part assumes 'unique_phases' and 'label_encoder' were defined in the same scope
# or you would need to re-initialize them based on your training data.
# For a complete deployment, save label_encoder using pickle or joblib.
# Example: import pickle
#          with open('label_encoder.pkl', 'wb') as f: pickle.dump(label_encoder, f)
#          with open('label_encoder.pkl', 'rb') as f: loaded_label_encoder = pickle.load(f)

# For simplicity, if you re-run the training cell first, unique_phases will be available.
# Otherwise, you would need to get the unique phases from the dataset.

# If `main()` has just been executed, `unique_phases` should be available in the global scope.
# If not, you might need to re-run the `main()` function or ensure `load_and_preprocess_data`
# and `get_all_phases` are called to define it.
# For robustness, let's ensure it's defined.
try:
    _ = unique_phases
except NameError:
    print("\n'unique_phases' not found in current scope. Rerunning data preprocessing to define it...")
    texts, _, _ = load_and_preprocess_data()
    unique_phases = get_all_phases(texts)
    print(f"Defined unique_phases: {unique_phases}")

# Re-initialize LabelEncoder with the correct classes
loaded_label_encoder = LabelEncoder()
loaded_label_encoder.fit(unique_phases)

def predict_phase(text: str) -> str:
    inputs = loaded_tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512)
    with torch.no_grad():
        outputs = loaded_model(**inputs)
    logits = outputs.logits
    prediction = torch.argmax(logits, dim=-1).item()
    predicted_phase = loaded_label_encoder.inverse_transform([prediction])[0]
    return predicted_phase

# Example Usage
example_text_1 = "Scanning the network for open ports and services using Nmap."
example_text_2 = "Gathering information about the target's employees from public sources."
example_text_3 = "Attempting to exploit a known vulnerability in a web application to gain a shell."
example_text_4 = "Deleting log files and clearing command history to remove traces of activity."
example_text_5 = "Setting up a persistent backdoor for future access."

print(f"\nPrediction for '{example_text_1}': {predict_phase(example_text_1)}")
print(f"Prediction for '{example_text_2}': {predict_phase(example_text_2)}")
print(f"Prediction for '{example_text_3}': {predict_phase(example_text_3)}")
print(f"Prediction for '{example_text_4}': {predict_phase(example_text_4)}")
print(f"Prediction for '{example_text_5}': {predict_phase(example_text_5)}")

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

matt8638_cybsec_training_path = kagglehub.dataset_download('matt8638/cybsec-training')

print('Data source import complete.')


In [ ]:
import json
import os
import logging
from typing import Dict, List, Optional, Tuple

import torch
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.preprocessing import LabelEncoder
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

# Setup Standard Logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# --- Helper functions (defined locally for scope safety) ---
def extract_pentest_phase(text: str) -> str:
    phases = {
        "reconnaissance": ["reconnaissance", "information gathering", "footprinting"],
        "scanning": ["scanning", "enumeration", "vulnerability assessment"],
        "gaining_access": ["exploitation", "gaining access", "privilege escalation"],
        "maintaining_access": ["persistence", "maintaining access", "backdoor"],
        "covering_tracks": ["covering tracks", "log cleaning", "evidence removal"],
    }
    text = text.lower()
    for phase, keywords in phases.items():
        if any(keyword in text for keyword in keywords):
            return phase
    return "other"

def normalize_text_value(value: Optional[str]) -> str:
    return str(value).strip() if value is not None else ""

def load_and_preprocess_data() -> Tuple[List[str], List[str], List[Dict[str, str]]]:
    texts, labels, qa_pairs = [], [], []
    if 'matt8638_cybsec_training_path' not in globals():
        raise NameError("Data path 'matt8638_cybsec_training_path' not found. Please run the Kaggle download cell.")

    target_dir = matt8638_cybsec_training_path
    for dirname, _, filenames in os.walk(target_dir):
        for filename in filenames:
            if filename.endswith('.json'):
                with open(os.path.join(dirname, filename), "r", encoding="utf-8") as f:
                    data_json = json.load(f)
                    for entry in data_json:
                        text_candidate = entry.get("description", "") or entry.get("question", "")
                        texts.append(normalize_text_value(text_candidate))
                        labels.append(extract_pentest_phase(text_candidate))
    return texts, labels, qa_pairs

def prepare_dataset(texts: List[str], labels: List[str], label_encoder: LabelEncoder, tokenizer: AutoTokenizer) -> Dataset:
    encoded_labels = label_encoder.transform(labels)
    encodings = tokenizer(texts, truncation=True, padding=True, max_length=512)
    return Dataset.from_dict({"input_ids": encodings["input_ids"], "attention_mask": encodings["attention_mask"], "labels": torch.tensor(encoded_labels)})

# --- Evaluation Logic using Hugging Face Model ---
hf_repo = "MattP30098638/PenTest-AI"
print(f"Loading model and tokenizer from Hugging Face: {hf_repo}")

try:
    eval_tokenizer = AutoTokenizer.from_pretrained(hf_repo)
    eval_model = AutoModelForSequenceClassification.from_pretrained(hf_repo)

    # Load and process data
    texts, labels, _ = load_and_preprocess_data()
    unique_phases = sorted(list(set(labels)))
    label_encoder = LabelEncoder().fit(unique_phases)

    # Split data
    # Removed 'stratify=encoded_labels' because some classes have only 1 member
    encoded_labels = label_encoder.transform(labels)
    _, test_texts, _, test_labels = train_test_split(
        texts,
        labels,
        test_size=0.2,
        random_state=42
    )

    test_dataset = prepare_dataset(test_texts, test_labels, label_encoder, eval_tokenizer)

    def compute_metrics(p):
        predictions = p.predictions.argmax(axis=1)
        accuracy = accuracy_score(p.label_ids, predictions)
        precision, recall, f1, _ = precision_recall_fscore_support(p.label_ids, predictions, average='weighted', zero_division=0)
        return {'accuracy': accuracy, 'f1': f1, 'precision': precision, 'recall': recall}

    trainer = Trainer(
        model=eval_model,
        args=TrainingArguments(output_dir="./eval_results", per_device_eval_batch_size=8, report_to="none"),
        eval_dataset=test_dataset,
        compute_metrics=compute_metrics
    )

    print("\nRunning Evaluation...")
    results = trainer.evaluate()
    for key, value in results.items():
        print(f"  {key}: {value:.4f}")
except Exception as e:
    print(f"Error during evaluation: {e}")

In [ ]:
import yaml

# Extract relevant metrics from the results dictionary
eval_metrics = {
    'eval_results': [
        {'key': 'loss', 'value': round(results.get('eval_loss', 0), 4)},
        {'key': 'accuracy', 'value': round(results.get('eval_accuracy', 0), 4)},
        {'key': 'f1', 'value': round(results.get('eval_f1', 0), 4)},
        {'key': 'precision', 'value': round(results.get('eval_precision', 0), 4)},
        {'key': 'recall', 'value': round(results.get('eval_recall', 0), 4)}
    ]
}

print("---\n# Copy this into your Hugging Face README.md metadata section\nmodel-index:\n- name: PenTest-AI\n  results:")
print(yaml.dump(eval_metrics, sort_keys=False, indent=4))
print("---")

In [ ]:
# Install huggingface_hub if you haven't already
!pip install huggingface_hub -qq

from huggingface_hub import notebook_login

# This will open a new browser tab or ask for your token in the output.
# You can get a token from: https://huggingface.co/settings/tokens
notebook_login()

In [ ]:
import os
import pickle

# Define your desired Hugging Face repository name
# This will create a new repository under your Hugging Face username
repo_name = "MattP30098638/PenTest-AI" # <--- CHANGE THIS TO A UNIQUE NAME

# Save the label_encoder
label_encoder_path = os.path.join("/content/model_output", "label_encoder.pkl")
with open(label_encoder_path, 'wb') as f:
    pickle.dump(loaded_label_encoder, f)
print(f"LabelEncoder saved to {label_encoder_path}")

# Push model and tokenizer to Hugging Face
# The 'loaded_model' and 'loaded_tokenizer' objects were created above
# You can also use `model.push_to_hub()` and `tokenizer.push_to_hub()` if you still have the original objects
# or if you reload them from the /content/model_output directory.

# For pushing the entire directory including the saved label_encoder
from huggingface_hub import HfApi

api = HfApi()

# Create a repository (if it doesn't exist)
# You can also set `private=True` if you want a private repo
api.create_repo(repo_id=repo_name, repo_type="model", exist_ok=True)

# Upload the contents of your model_output directory
api.upload_folder(
    folder_path="/content/model_output",
    repo_id=repo_name,
    repo_type="model",
    commit_message="Upload fine-tuned RoBERTa-large model and label encoder"
)

print(f"\nModel and tokenizer, along with label_encoder, uploaded to Hugging Face!\n")
print(f"You can find it here: https://huggingface.co/{api.whoami()['name']}/{repo_name}")

In [ ]:
import shutil
import os

output_dir = '/content/model_output'
zip_path = '/content/model_output.zip'

if os.path.exists(output_dir):
    shutil.make_archive(zip_path.replace('.zip', ''), 'zip', output_dir)
    print(f"Successfully created {zip_path}. You can now download this file.")
else:
    print(f"Error: The directory {output_dir} does not exist.")